# Bi-Mamba 55M — Chinese ↔ Vietnamese translation, end-to-end on Colab

Notebook chạy **toàn bộ pipeline** từ A đến Z: tải dữ liệu → train tokenizer → train Bi-Mamba 55M → đánh giá SacreBLEU → dịch demo.

**Yêu cầu:** Runtime → GPU (T4 / L4 / A100 đều OK). RAM ≥ 12 GB.

> Mặc định subsample còn 200 K cặp để chạy đầy đủ trên T4 free trong khoảng 1.5–2 giờ. Bỏ subsample nếu muốn train trên toàn bộ corpus.


## 1. Mount Google Drive (tuỳ chọn)

Để lưu checkpoint vĩnh viễn. Nếu không cần, bạn có thể bỏ qua ô này.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Skipping drive mount:', e)


## 2. Clone repo

Đang clone từ **`ChauDucToan/NLP_DHM`** — đổi `REPO_URL` nếu bạn fork sang chỗ khác.

In [ ]:
import os
REPO_URL = 'https://github.com/ChauDucToan/NLP_DHM.git'
REPO_DIR = '/content/NLP_DHM'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only || true
%cd $REPO_DIR
!pwd && ls


## 3. Cài đặt dependencies

`mamba-ssm` + `causal-conv1d` (CUDA fast-path) thường **không build được trên Colab** vì wheel CUDA không khớp — không sao, repo có sẵn **fallback PyTorch thuần** sẽ chạy đúng kết quả, chỉ chậm hơn vài lần.

In [ ]:
!pip install -q -r requirements.txt
# Optional CUDA kernels — silently fall back if they don't build.
!pip install -q causal-conv1d mamba-ssm 2>/dev/null || echo 'mamba-ssm wheels not available; using pure-PyTorch Mamba (slower but identical results).'
import torch, sys
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
sys.path.insert(0, 'src')


## 4. Configuration

Sửa các giá trị bên dưới nếu muốn chạy nhanh / lâu hơn. Mặc định: subsample 200 K cặp, max_steps 30 K — phù hợp T4 trong 1.5–2 giờ.

In [ ]:
import yaml
from pathlib import Path

cfg_path = Path('configs/bi_mamba_55m.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

# === Data knobs ===
cfg['data']['preset']           = 'small'      # tiny | small | medium | large
cfg['data']['max_train_pairs']  = 200_000      # set to None for full corpus
cfg['data']['max_valid_pairs']  = 2_000
cfg['data']['max_test_pairs']   = 2_000

# === Train knobs ===
cfg['train']['max_steps']   = 30_000
cfg['train']['batch_size']  = 32        # bump to 64+ on A100 / L4
cfg['train']['amp_dtype']   = 'bf16'    # 'fp16' on T4 if bf16 unsupported
cfg['train']['eval_every']  = 2_000
cfg['train']['save_every']  = 2_000
cfg['train']['log_every']   = 50
cfg['train']['num_workers'] = 2

Path('configs/_colab.yaml').write_text(
    yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True)
)
print('Saved configs/_colab.yaml')


## 5. Tải + chuẩn bị dữ liệu (OPUS zh-vi)

`prepare_data.py` tải trực tiếp từ **OPUS** (`object.pouta.csc.fi`). Không cần Hugging Face dataset config — `Helsinki-NLP/opus-100` thực ra **không có** cặp zh-vi.

Presets:

| preset    | sources                                                  | ~ pairs   | size  |
|-----------|----------------------------------------------------------|-----------|-------|
| `tiny`    | TED2020                                                  | 50 K      | 1 MB  |
| `small`   | TED2020 + WikiMatrix + bible-uedin                       | 200 K     | 25 MB |
| `medium`  | small + OpenSubtitles vi-zh_cn                           | 3 M       | 65 MB |
| `large`   | medium + NLLB                                            | 30 M      | 700 MB|

Bạn cũng có thể truyền `--custom-jsonl path/to/your.jsonl` (mỗi dòng `{"zh": "...", "vi": "..."}`).

In [ ]:
!python scripts/prepare_data.py --config configs/_colab.yaml --preset {cfg['data']['preset']}


## 6. Train SentencePiece tokenizer (chia sẻ chung zh+vi)

In [ ]:
!python scripts/train_tokenizer.py --config configs/_colab.yaml


## 7. Khởi tạo model + kiểm tra số tham số

Mục tiêu thiết kế: ~55 M tham số. Thực tế: **54.63 M**.

In [ ]:
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.utils import count_parameters, human_format

model = BiMambaTranslator(ModelConfig(**cfg['model']))
n = count_parameters(model)
print(f'Bi-Mamba parameters: {human_format(n)}  ({n:,})')


## 8. Train end-to-end

Checkpoint sẽ được ghi vào `runs/bi_mamba_55m/`. Có thể dừng và bật lại notebook với `--resume <ckpt>` để train tiếp.

In [ ]:
!python scripts/train.py --config configs/_colab.yaml


## 9. Đánh giá SacreBLEU + chrF (cả hai chiều zh→vi và vi→zh)

In [ ]:
!python scripts/evaluate.py --config configs/_colab.yaml --num-samples 1000 --beam-size 4


## 10. Demo dịch

In [ ]:
import torch
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.tokenizer import Tokenizer
from bi_mamba_mt.translate import translate_batch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(f"{cfg['train']['output_dir']}/latest.pt", map_location='cpu')
model = BiMambaTranslator(ModelConfig(**ckpt['model_cfg'])).to(device)
model.load_state_dict(ckpt['model']); model.eval()
tok = Tokenizer(f"{cfg['data']['tokenizer_dir']}/spm.model")

zh_samples = [
    '你好，世界。',
    '今天的天气很好。',
    '这本书我已经读完了。',
]
vi_samples = [
    'Xin chào thế giới.',
    'Hôm nay trời thật đẹp.',
    'Tôi đã đọc xong cuốn sách này.',
]

print('--- zh → vi ---')
for s, t in zip(zh_samples,
                translate_batch(model, tok, zh_samples, 'zh2vi',
                                beam_size=4, device=device)):
    print(f'  {s!r} -> {t!r}')

print('--- vi → zh ---')
for s, t in zip(vi_samples,
                translate_batch(model, tok, vi_samples, 'vi2zh',
                                beam_size=4, device=device)):
    print(f'  {s!r} -> {t!r}')


## 11. (Tuỳ chọn) Lưu checkpoint vào Drive

In [ ]:
import shutil, os
DRIVE_DIR = '/content/drive/MyDrive/bi_mamba_55m'
if os.path.isdir('/content/drive/MyDrive'):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for fname in ('latest.pt', 'final.pt', 'config.yaml'):
        src = f"{cfg['train']['output_dir']}/{fname}"
        if os.path.isfile(src):
            shutil.copy(src, DRIVE_DIR)
            print('Copied', src, '->', DRIVE_DIR)
    tok_dst = f'{DRIVE_DIR}/tokenizer'
    os.makedirs(tok_dst, exist_ok=True)
    for fname in ('spm.model', 'spm.vocab'):
        src = f"{cfg['data']['tokenizer_dir']}/{fname}"
        if os.path.isfile(src):
            shutil.copy(src, tok_dst)
    print('Done. Files in', DRIVE_DIR)
else:
    print('Drive not mounted; skipping.')
